# Neural Network Activation Functions and Dynamic Dispatch

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Artificial neural networks require non-linear activation functions applied after each affine transformation ($z = Wx + b$). Without non-linearities, any deep stack of linear transformations collapses mathematically into a single linear map:
$$W_k (W_{k-1} \dots (W_1 x + b_1) \dots + b_{k-1}) + b_k = W^* x + b^*$$

Activation functions provide the non-linear bending capacity that enables neural networks to act as universal function approximators.

### Objectives
1. Formulate Binary Step, Sigmoid (Logistic), and Exponential Linear Unit (ELU) activations.
2. Implement numerical parsing and robust input string verification.
3. Implement a dynamic activation function dispatcher with defensive error handling.
4. Visualize and compare the mathematical curves of the activation functions.


## 1. Activation Formulations

1. **Binary Step Function (Heaviside):**
   $$f(x) = \begin{cases} 1 & \text{if } x \ge 0 \\ 0 & \text{if } x < 0 \end{cases}$$
   Historical relevance: Used in Frank Rosenblatt's Perceptron (1958). Cannot be used in gradient-based backpropagation because its derivative is zero everywhere (except at $x=0$ where it is undefined).

2. **Sigmoid Function:**
   $$\sigma(x) = \frac{1}{1 + e^{-x}}$$
   Properties: Smooth S-shaped curve mapping $\mathbb{R} \to (0, 1)$. Frequently used for binary classification output probabilities. Drawbacks include vanishing gradients for large $|x|$ and non-zero-centered outputs.

3. **Exponential Linear Unit (ELU):**
   $$f(x) = \begin{cases} x & \text{if } x \ge 0 \\ \alpha (e^x - 1) & \text{if } x < 0 \end{cases}$$
   Properties: For $x \ge 0$, acts like standard ReLU, avoiding saturation. For $x < 0$, saturates smoothly to $-\alpha$, pushing mean activations closer to zero and improving gradient flow. In this lab, we use $\alpha = 0.01$.


In [ ]:
import math


def is_number(n: str) -> bool:
    """Returns True if string n can be parsed as a float, False otherwise."""
    try:
        float(n)
        return True
    except ValueError:
        return False


def binary_step(x: float) -> float:
    """Computes the binary step function."""
    return 1.0 if x >= 0.0 else 0.0


def sigmoid(x: float) -> float:
    """Computes the sigmoid function using math.e."""
    return 1.0 / (1.0 + (math.e ** (-x)))


def elu(x: float, alpha: float = 0.01) -> float:
    """Computes the Exponential Linear Unit (ELU) with parameter alpha."""
    if x >= 0.0:
        return float(x)
    return float(alpha * ((math.e**x) - 1.0))

## 2. Dispatcher Implementation with Defensive Validation

The lab specification requires:
1. Verify `x` with `is_number(x)`. If invalid, print `'x must be a number'` and return.
2. Check `function_name` against supported activations (`'binary'`, `'sigmoid'`, `'elu'`). If unsupported, print `'{function_name} is not supported'` and return.
3. Cast `x` to `float`, calculate the result, and print `'{function_name}: f({x}) = {result}'`.


In [ ]:
def evaluate_activation(x_str: str, function_name: str) -> None:
    """Parses inputs, validates values, evaluates the requested activation, and displays results."""
    if not is_number(x_str):
        print("x must be a number")
        return

    name = function_name.strip().lower()
    valid_functions = {"binary", "sigmoid", "elu"}
    if name not in valid_functions:
        print(f"{function_name} is not supported")
        return

    x = float(x_str)
    if name == "binary":
        result = binary_step(x)
    elif name == "sigmoid":
        result = sigmoid(x)
    elif name == "elu":
        result = elu(x, alpha=0.01)

    print(f"{name}: f({x})={result}")

## 3. Testing Execution Paths and Edge Cases


In [ ]:
print("=== Valid Executions ===")
evaluate_activation("3", "binary")
evaluate_activation("-2", "binary")
evaluate_activation("0", "sigmoid")
evaluate_activation("1.5", "elu")
evaluate_activation("-1.5", "elu")

print("\n=== Invalid Value Case ===")
evaluate_activation("abc", "sigmoid")

print("\n=== Unsupported Function Case ===")
evaluate_activation("2.0", "relu")
evaluate_activation("1.0", "tanh")

## 4. Visualizing Activation Functions

We plot Binary Step, Sigmoid, and ELU across the domain $x \in [-5, 5]$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(-5.0, 5.0, 300)
y_step = [binary_step(x) for x in xs]
y_sig = [sigmoid(x) for x in xs]
y_elu = [elu(x, alpha=0.5) for x in xs]  # Using alpha=0.5 for visual clarity

plt.figure(figsize=(10, 5))
plt.plot(xs, y_step, label="Binary Step", linestyle="--", color="#d9534f")
plt.plot(xs, y_sig, label="Sigmoid", linewidth=2, color="#0275d8")
plt.plot(xs, y_elu, label="ELU (alpha=0.5)", linewidth=2, color="#5cb85c")
plt.axhline(0, color="black", linewidth=0.8, linestyle=":")
plt.axvline(0, color="black", linewidth=0.8, linestyle=":")
plt.title("Neural Network Activation Functions Comparison")
plt.xlabel("Input x")
plt.ylabel("Activation f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()